# 🔗 Day 02a: SQL JOINs — INNER, LEFT, RIGHT, FULL, CROSS, SELF

---

## 🎯 What You'll Master Today
- INNER JOIN, LEFT/RIGHT OUTER JOIN, FULL OUTER JOIN
- CROSS JOIN (Cartesian product)
- SELF JOIN (manager hierarchy)
- Multi-table joins and join gotchas

---

## 🏗️ The Puzzle Analogy

> Imagine two puzzle boxes. INNER JOIN keeps only pieces that fit together.
> LEFT JOIN keeps ALL pieces from the left box + matching right pieces.
> FULL OUTER JOIN keeps everything from both boxes — unmatched pieces get NULLs.

```
╔══════════════════════════════════════════════════════════════════╗
║                    JOIN VISUAL GUIDE                             ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  Table A                Table B                                  ║
║  ┌─────┐               ┌─────┐                                  ║
║  │ 1   │               │ 1   │                                  ║
║  │ 2   │               │ 2   │                                  ║
║  │ 3   │               │ 4   │                                  ║
║  └─────┘               └─────┘                                  ║
║                                                                  ║
║  INNER:  {1,2}       — only matches                             ║
║  LEFT:   {1,2,3}     — all A + matching B (3→NULL)              ║
║  RIGHT:  {1,2,4}     — all B + matching A (4→NULL)              ║
║  FULL:   {1,2,3,4}   — everything (3→NULL, 4→NULL)             ║
║  CROSS:  {1×1,1×2,1×4, 2×1,2×2,2×4, 3×1,3×2,3×4} — 9 rows   ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# Setup: Two related tables
# ============================================

employees = [
    {"id": 1, "name": "Alice",   "dept_id": 10, "mgr_id": None},
    {"id": 2, "name": "Bob",     "dept_id": 10, "mgr_id": 1},
    {"id": 3, "name": "Charlie", "dept_id": 20, "mgr_id": 1},
    {"id": 4, "name": "Diana",   "dept_id": 20, "mgr_id": 3},
    {"id": 5, "name": "Eve",     "dept_id": 30, "mgr_id": 1},
    {"id": 6, "name": "Frank",   "dept_id": None, "mgr_id": None},  # no department!
]

departments = [
    {"id": 10, "name": "Engineering", "budget": 500000},
    {"id": 20, "name": "Marketing",   "budget": 300000},
    {"id": 40, "name": "Legal",       "budget": 200000},  # no employees!
]

def print_table(rows, columns=None):
    if not rows:
        print("  (empty)")
        return
    if columns is None:
        columns = list(rows[0].keys())
    widths = {c: max(len(str(c)), max(len(str(r.get(c, ''))) for r in rows)) for c in columns}
    header = "  " + " | ".join(f"{c:<{widths[c]}}" for c in columns)
    print(header)
    print("  " + "-+-".join("-" * widths[c] for c in columns))
    for r in rows:
        print("  " + " | ".join(f"{str(r.get(c, '')):<{widths[c]}}" for c in columns))

print("  Employees:")
print_table(employees)
print("\n  Departments:")
print_table(departments)
print("\n  Key: Frank has NO dept. Legal has NO employees. Eve's dept 30 doesn't exist!")

In [ ]:
# ============================================
# 1. INNER JOIN — only matching rows
# ============================================

# SQL: SELECT e.name, d.name AS dept FROM employees e
#      INNER JOIN departments d ON e.dept_id = d.id;

print("  INNER JOIN employees ON dept_id = departments.id")
dept_lookup = {d["id"]: d for d in departments}
result = []
for e in employees:
    if e["dept_id"] in dept_lookup:
        result.append({"emp_name": e["name"], "dept_name": dept_lookup[e["dept_id"]]["name"]})
print_table(result)
print(f"\n  → {len(result)} rows. Frank (NULL dept), Eve (dept 30 missing), Legal (no emps) excluded")

In [ ]:
# ============================================
# 2. LEFT JOIN — all left + matching right
# ============================================

# SQL: SELECT e.name, d.name AS dept FROM employees e
#      LEFT JOIN departments d ON e.dept_id = d.id;

print("  LEFT JOIN: all employees, even without departments")
result = []
for e in employees:
    d = dept_lookup.get(e["dept_id"])
    result.append({"emp_name": e["name"], "dept_name": d["name"] if d else "NULL"})
print_table(result)
print("\n  → Frank & Eve show NULL dept. Legal still excluded (not in LEFT table)")

print("\n  LEFT JOIN to find 'orphans' (employees with no valid dept):")
print("  SQL: SELECT e.name FROM employees e LEFT JOIN departments d ON e.dept_id = d.id WHERE d.id IS NULL;")
orphans = [r for r in result if r["dept_name"] == "NULL"]
print_table(orphans)

In [ ]:
# ============================================
# 3. RIGHT JOIN & FULL OUTER JOIN
# ============================================

# SQL: SELECT e.name, d.name FROM employees e
#      RIGHT JOIN departments d ON e.dept_id = d.id;

print("  RIGHT JOIN: all departments, even without employees")
emp_by_dept = {}
for e in employees:
    emp_by_dept.setdefault(e["dept_id"], []).append(e)

result = []
for d in departments:
    emps = emp_by_dept.get(d["id"], [None])
    for e in emps:
        result.append({"emp_name": e["name"] if e else "NULL", "dept_name": d["name"]})
print_table(result)
print("\n  → Legal shows with NULL emp. Frank & Eve excluded (not in RIGHT table)")

# FULL OUTER JOIN
print("\n  FULL OUTER JOIN: ALL employees + ALL departments")
result = []
matched_depts = set()
for e in employees:
    d = dept_lookup.get(e["dept_id"])
    if d:
        matched_depts.add(d["id"])
    result.append({"emp_name": e["name"], "dept_name": d["name"] if d else "NULL"})

for d in departments:
    if d["id"] not in matched_depts:
        result.append({"emp_name": "NULL", "dept_name": d["name"]})

print_table(result)
print("\n  → Everyone appears. Unmatched sides get NULLs.")

In [ ]:
# ============================================
# 4. CROSS JOIN & SELF JOIN
# ============================================

# CROSS JOIN: every combination
# SQL: SELECT e.name, d.name FROM employees e CROSS JOIN departments d;
print("  CROSS JOIN: |employees| × |departments| =", len(employees), "×", len(departments), "=", len(employees)*len(departments), "rows")
cross = [{"emp": e["name"], "dept": d["name"]} for e in employees for d in departments]
print_table(cross[:6])  # show first 6
print(f"  ... ({len(cross)} total rows)\n")

# SELF JOIN: employee → manager hierarchy
# SQL: SELECT e.name AS employee, m.name AS manager
#      FROM employees e LEFT JOIN employees m ON e.mgr_id = m.id;
print("  SELF JOIN: employee → manager")
emp_lookup = {e["id"]: e for e in employees}
result = []
for e in employees:
    mgr = emp_lookup.get(e["mgr_id"])
    result.append({"employee": e["name"], "manager": mgr["name"] if mgr else "(none)"})
print_table(result)
print("\n  💡 SELF JOIN is just a regular join where both sides are the same table")

In [ ]:
# ============================================
# 5. JOIN Pattern Map + Gotchas
# ============================================

print("""
  ┌────────────────┬──────────────────┬──────────────────────────────────────┐
  │ Join Type       │ Returns          │ Use Case                             │
  ├────────────────┼──────────────────┼──────────────────────────────────────┤
  │ INNER JOIN      │ matches only     │ Get employees WITH departments       │
  │ LEFT JOIN       │ all left + match │ Find orphans (NULL on right)         │
  │ RIGHT JOIN      │ all right + match│ Find empty depts (NULL on left)      │
  │ FULL OUTER      │ everything       │ Reconciliation / data audit          │
  │ CROSS JOIN      │ cartesian product│ Generate combinations (sizes×colors) │
  │ SELF JOIN       │ same table × 2   │ Hierarchies, compare rows in table   │
  └────────────────┴──────────────────┴──────────────────────────────────────┘

  ⚠️ JOIN GOTCHAS:
  ┌─────────────────────────────────────────────────────────────────┐
  │ 1. JOIN on NULLs: NULL = NULL → NOT matched (use COALESCE)    │
  │ 2. Duplicate rows: 1:N join multiplies rows. Always check!    │
  │ 3. Missing ON clause: becomes CROSS JOIN (accidental!)        │
  │ 4. JOIN vs subquery: JOIN for combining, subquery for filters │
  │ 5. MySQL: No FULL OUTER JOIN → use LEFT UNION RIGHT          │
  └─────────────────────────────────────────────────────────────────┘
""")

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | INNER vs LEFT JOIN? | INNER: only matches. LEFT: all left rows + NULLs for non-matches |
| 2 | How to find rows in A not in B? | LEFT JOIN + WHERE b.id IS NULL (anti-join pattern) |
| 3 | Can JOIN produce more rows than input? | Yes! 1:N or M:N joins multiply rows |
| 4 | JOIN vs subquery performance? | JOIN is usually faster — optimizer can choose best plan |
| 5 | Does NULL = NULL match in joins? | No! NULL = NULL evaluates to NULL (falsy). Use IS NULL |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • INNER = intersection, LEFT = all left + match         │
## │  • Anti-join pattern: LEFT JOIN + WHERE right IS NULL    │
## │  • SELF JOIN: same table aliased twice (hierarchies)     │
## │  • CROSS JOIN: cartesian product (rows multiply!)        │
## │  • NULL never matches in ON clause — watch for this!     │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Subqueries & CTEs** — Scalar, Correlated, EXISTS, WITH